## DataFrames
- PySpark Dataframe
- Reading The Dataset
- Checking the Data types of the Column(Schema)
- Selecting columns and indexing
- checking the describe function (and the similarities to pandas)
- Adding + Dropping Columns
- Renaming Columns

#### PySpark DataFrame

In [31]:
from pyspark.sql import SparkSession

In [32]:
spark = SparkSession.builder.appName('Dataframe').getOrCreate()

#### Reading The Dataset

In [33]:
# Reading the Dataset
df_spark = spark.read.option("header", "true").csv("spark_basics.csv")
df_spark.show()

+------+---+----------+
|  Name|Age|Experience|
+------+---+----------+
|  Gary| 22|         4|
|Andrew| 21|         8|
|  Ryan| 22|        12|
+------+---+----------+



#### Checking Data Types Of The Columns

In [34]:
df_spark.printSchema()

# by default, every variable type is set to string, so even values that should logically be integers are automatically cast to strings
# however as see in the example below, both string and experience, which are integers in the csv are converted to strings as well

root
 |-- Name: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- Experience: string (nullable = true)



In [35]:
# Alternate Method to Read Data: Adding an option in .csv() setting inferSchema to True
# in order to have other variable types other than just string

df_spark = spark.read.option("header", 'true').csv("spark_basics.csv", inferSchema=True)
df_spark.printSchema()

# now age and experience are both doubles

root
 |-- Name: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Experience: integer (nullable = true)



In [36]:
df_spark = spark.read.csv("spark_basics.csv", header=True, inferSchema=True)
df_spark.show()
# this is another method of calling csv's so that they still retain both the original headers and accurate data types for each column

+------+---+----------+
|  Name|Age|Experience|
+------+---+----------+
|  Gary| 22|         4|
|Andrew| 21|         8|
|  Ryan| 22|        12|
+------+---+----------+



In [37]:
df_spark.printSchema()

root
 |-- Name: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Experience: integer (nullable = true)



#### Selecting Columns and Indexing

In [38]:
df_spark.columns
# spark columns functions works the same as pandas columns function to quickly find all the columns in the data frame

['Name', 'Age', 'Experience']

In [39]:
df_spark.head(3)
# when using .head function on spark datasets, to get the top X number of records, it is returned in a list format
# similar concept to pandas, but returns the values in a different format

[Row(Name='Gary', Age=22, Experience=4),
 Row(Name='Andrew', Age=21, Experience=8),
 Row(Name='Ryan', Age=22, Experience=12)]

In [43]:
# selecting columns is different to a pandas function
# if you were to do it the exact same way you would get an error

df_spark["Name"]
# calling .show() would result in an TypeError: 'Column' object is not callable
# so in order to see the values in the column, the .Select() function is required

Column<'Name'>

Selecting A Single Column

In [40]:
# Selecting a column in a PySpark data frame: Using the .select(column_name) function
# resulting in another pyspark.sql.dataframe

df_spark.select('Name').show()

+------+
|  Name|
+------+
|  Gary|
|Andrew|
|  Ryan|
+------+



Selecting Multiple Columns

In [42]:
# selecting multiple columns: is the same as pandas: passing through a list of the column names
# [column_name_1, column_name_2, column_name_3, ..., column_name_N]
df_spark.select(['Name', 'Experience']).show()

+------+----------+
|  Name|Experience|
+------+----------+
|  Gary|         4|
|Andrew|         8|
|  Ryan|        12|
+------+----------+



#### PySpark Describe() Method Options

In [44]:
df_spark.dtypes
# does not provide what the columns are stored in but, provides a summary of the data types of all columns

[('Name', 'string'), ('Age', 'int'), ('Experience', 'int')]

In [45]:
df_spark.describe()
# provides a summary of the data types of all columns as well as what they are stored in: a DataFrame

DataFrame[summary: string, Name: string, Age: string, Experience: string]

In [46]:
df_spark.describe().show()

# mean and stddev for name are both null since they are both calculated numerically, and name is a string, so there is no way to calculate the mean of strings

+-------+------+------------------+----------+
|summary|  Name|               Age|Experience|
+-------+------+------------------+----------+
|  count|     3|                 3|         3|
|   mean|  NULL|21.666666666666668|       8.0|
| stddev|  NULL|0.5773502691896258|       4.0|
|    min|Andrew|                21|         4|
|    max|  Ryan|                22|        12|
+-------+------+------------------+----------+



#### Adding And Dropping Columns In PySpark DataFrame

Adding Columns

In [54]:
# Adding Columns in data frames: is not an inplace operation, meaning that for changes to take place, the change has to be assigned to a variable
# df_spark.withColumn('new_column_name', 'new_column_values')
df_spark = df_spark.withColumn('Experience After 2 Years', df_spark['Experience']*1.5)
# is similar concept to adding columns in pandas, with the new column name, and the new values, that can be a previous column with any additions

df_spark.withColumn('Experience After 2 Years', df_spark['Experience']*1.5)
# now you can see that in the output below there is a new column added: Experience After 2 Years

DataFrame[Name: string, Age: int, Experience: int, Experience After 2 Years: double]

In [52]:
df_spark.show()

+------+---+----------+------------------------+
|  Name|Age|Experience|Experience After 2 Years|
+------+---+----------+------------------------+
|  Gary| 22|         4|                     6.0|
|Andrew| 21|         8|                    12.0|
|  Ryan| 22|        12|                    18.0|
+------+---+----------+------------------------+



Dropping Columns

In [56]:
# Dropping Columns In PySpark: is very similar to pandas
df_spark.drop('Experience After 2 Years').show()

# as can be seen in the data below, the column is no longer there

+------+---+----------+
|  Name|Age|Experience|
+------+---+----------+
|  Gary| 22|         4|
|Andrew| 21|         8|
|  Ryan| 22|        12|
+------+---+----------+



In [58]:
df_spark = df_spark.drop('Experience After 2 Years')
df_spark.show()
# assigning df_spark to the dropped column expression now results in a dataframe without the column

+------+---+----------+
|  Name|Age|Experience|
+------+---+----------+
|  Gary| 22|         4|
|Andrew| 21|         8|
|  Ryan| 22|        12|
+------+---+----------+



#### Renaming Column Names

In [59]:
# is another function: df_example.withColumnRenamed('existing_column_name', 'new_preferred_column_name')

df_spark.withColumnRenamed('Name', 'New Name').show()
# as can be seen below, now the column has been renamed, although, for the change to be recorded, it has to be assigned to another value

+--------+---+----------+
|New Name|Age|Experience|
+--------+---+----------+
|    Gary| 22|         4|
|  Andrew| 21|         8|
|    Ryan| 22|        12|
+--------+---+----------+

